In [1]:
# ============================================================
# CELL 1 — CHECK GPU + INSTALL PACKAGES
# ============================================================

import subprocess, sys, os

# Install extra packages (seaborn and scikit-learn are pre-installed on Kaggle)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=False)

import tensorflow as tf
import numpy as np

print('=' * 60)
print('  ENVIRONMENT CHECK')
print('=' * 60)
print(f'  TensorFlow   : {tf.__version__}')
print(f'  NumPy        : {np.__version__}')
print(f'  Python       : {sys.version.split()[0]}')

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'\n  ✅ GPU detected: {gpus[0].name}')
    print(f'     Training will be FAST (~30 min for 50 epochs)')
else:
    print('\n  ⚠️  No GPU detected — go to Settings → Accelerator → GPU T4 x1')

# Check dataset path exists
DATASET_BASE = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'
if os.path.exists(DATASET_BASE):
    print(f'\n  ✅ Dataset found at: {DATASET_BASE}')
    # List contents to understand folder structure
    for item in sorted(os.listdir(DATASET_BASE))[:5]:
        print(f'     └── {item}')
    print(f'     ... (showing first 5 items)')
else:
    print(f'\n  ❌ Dataset NOT found at: {DATASET_BASE}')
    print('     → Click "+ Add Data" on right panel')
    print('     → Search: plantvillage-dataset (by abdallahalidev)')
    print('     → Click Add, then re-run this cell')

print('\n' + '=' * 60)
print('  CELL 1 DONE — Ready to proceed')
print('=' * 60)

2026-03-10 18:40:40.231002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773168040.455012      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773168040.510781      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773168041.020905      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773168041.020945      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773168041.020948      55 computation_placer.cc:177] computation placer alr

  ENVIRONMENT CHECK
  TensorFlow   : 2.19.0
  NumPy        : 2.0.2
  Python       : 3.12.12

  ✅ GPU detected: /physical_device:GPU:0
     Training will be FAST (~30 min for 50 epochs)

  ✅ Dataset found at: /kaggle/input/datasets/emmarex/plantdisease/PlantVillage
     └── Pepper__bell___Bacterial_spot
     └── Pepper__bell___healthy
     └── Potato___Early_blight
     └── Potato___Late_blight
     └── Potato___healthy
     ... (showing first 5 items)

  CELL 1 DONE — Ready to proceed


In [2]:
"""
improved_pest_detection.py
===========================
UPDATED VERSION - Aligned with Colab Notebook + Full Inference Support

Based on Research Papers:
  Paper 1: Nyakuri et al. (2025) - AI and IoT-powered edge device
  Paper 2: Khirade & Patil (2015) - Plant Disease Detection Using Image Processing

WHAT THIS FILE DOES:
  - Used by Colab notebook (Step 4) for training
  - Used for inference on trained model (Colab/RPi/ESP32)
  - All 3 improvement steps from research included

HOW IT IS USED:
  1. Colab Training  -> PestDetectionSystem + TinyLiteNet.build_model()
  2. Colab Inference -> show_predictions(), plot_training_history()
  3. Raspberry Pi    -> predict_from_image_path()
  4. ESP32 server    -> predict_from_array()
"""

import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# STEP 1: ENHANCED PRE-PROCESSING  (From Paper 2 - Khirade & Patil 2015)
# ============================================================================

class ImagePreprocessor:
    """
    Complete pre-processing pipeline from Paper 2.

    Steps applied in order:
      1. Resize to target_size
      2. Gaussian blur      -> removes camera noise
      3. Histogram equaliz  -> improves contrast in any lighting
      4. Contrast enhance   -> makes disease features clearer
    """

    @staticmethod
    def noise_removal(image, kernel_size=5):
        """Gaussian blur to remove noise"""
        return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

    @staticmethod
    def histogram_equalization(image):
        """
        Histogram equalization on Y channel (YUV space).
        Paper 2: 'distributes the intensities of the images'
        Keeps colour, only changes brightness distribution.
        """
        img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
        img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
        return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

    @staticmethod
    def enhance_contrast(image, alpha=1.3, beta=0):
        """
        Contrast enhancement.
        alpha > 1 increases contrast.
        """
        return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    @staticmethod
    def preprocess_image(image, target_size=(224, 224)):
        """
        Full pipeline called for every image before CNN.
        Works on a numpy BGR array (same as cv2.imread output).
        """
        image = cv2.resize(image, target_size)
        image = ImagePreprocessor.noise_removal(image)
        image = ImagePreprocessor.histogram_equalization(image)
        image = ImagePreprocessor.enhance_contrast(image)
        return image


# ============================================================================
# STEP 2: SEGMENTATION LAYER  (From Paper 2 - Khirade & Patil 2015)
# ============================================================================

class ImageSegmenter:
    """
    Implements three segmentation methods from Paper 2:
      1. K-means clustering (k=3)  -> separates healthy/diseased/background
      2. Green pixel masking        -> removes healthy green tissue
      3. Otsu thresholding          -> binary mask of diseased region
      + Canny edge detection        -> highlights lesion boundaries
    """

    @staticmethod
    def kmeans_segmentation(image, k=3):
        """
        K-means clustering.
        Paper 2: 'K-means clustering is more accurate than other methods'
        k=3 -> healthy tissue, diseased tissue, background
        """
        pixel_values = np.float32(image.reshape((-1, 3)))
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
        _, labels, centers = cv2.kmeans(
            pixel_values, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
        )
        centers  = np.uint8(centers)
        segmented = centers[labels.flatten()].reshape(image.shape)
        return segmented

    @staticmethod
    def green_pixel_masking(image):
        """
        Remove healthy (green) pixels so CNN focuses on diseased areas.
        Paper 2: 'green pixels masked and removed'
        """
        hsv           = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        lower_green   = np.array([35, 40, 40])
        upper_green   = np.array([85, 255, 255])
        green_mask    = cv2.inRange(hsv, lower_green, upper_green)
        diseased_mask = cv2.bitwise_not(green_mask)
        masked        = cv2.bitwise_and(image, image, mask=diseased_mask)
        return masked, diseased_mask

    @staticmethod
    def otsu_thresholding(image):
        """
        Otsu automatic thresholding.
        Paper 2: 'creates binary images from grey-level images'
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary

    @staticmethod
    def segment_image(image):
        """
        Full segmentation pipeline.
        Returns dict so every stage can be visualised separately.
        """
        segmented          = ImageSegmenter.kmeans_segmentation(image, k=3)
        masked, mask       = ImageSegmenter.green_pixel_masking(segmented)
        binary             = ImageSegmenter.otsu_thresholding(masked)
        gray               = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
        edges              = cv2.Canny(gray, 50, 150)

        return {
            'original':  image,
            'segmented': segmented,
            'masked':    masked,
            'binary':    binary,
            'edges':     edges,
            'mask':      mask
        }


# ============================================================================
# STEP 3: TINY-LITENET CNN  (From Paper 1 - Nyakuri et al. 2025)
# ============================================================================

class TinyLiteNet:
    """
    Lightweight CNN model.
    Paper 1 specs: 1.2 MB, 1.48 M parameters, 16 ms inference on RPi5.
    Architecture: MobileNetV2 inspired + 6 Squeeze-Excitation (SE) blocks.
    """

    @staticmethod
    def squeeze_excitation_block(x, ratio=16):
        """
        SE block - recalibrates channel-wise feature responses.
        Paper 1: '6 SE depthwise blocks to enhance feature representation'
        """
        channels = x.shape[-1]
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Dense(max(1, channels // ratio), activation='relu')(se)
        se = layers.Dense(channels, activation='sigmoid')(se)
        se = layers.Reshape((1, 1, channels))(se)
        return layers.Multiply()([x, se])

    @staticmethod
    def depthwise_block(x, filters, stride=1):
        """Depthwise-separable conv block (MobileNetV2 style)."""
        x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)
        x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)
        return x

    @staticmethod
    def build_model(input_shape=(224, 224, 3), num_classes=9):
        """
        Build Tiny-LiteNet.
        Target: ~1.2 MB, ~1.48 M parameters.
        """
        inputs = layers.Input(shape=input_shape, name='input_image')

        # Stem conv
        x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)

        # 6 Depthwise + SE blocks  (Paper 1: 6 SE depthwise blocks)
        configs = [(64,1), (128,2), (128,1), (256,2), (256,1), (512,2)]
        for filters, stride in configs:
            x = TinyLiteNet.depthwise_block(x, filters, stride)
            x = TinyLiteNet.squeeze_excitation_block(x)

        # Classification head
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

        model = models.Model(inputs, outputs, name='TinyLiteNet')
        return model

    @staticmethod
    def model_info(model):
        """Print model size and parameter count."""
        params  = model.count_params()
        size_mb = (params * 4) / (1024 * 1024)
        print(f"\n{'='*60}")
        print(f"  Model        : {model.name}")
        print(f"  Parameters   : {params:,}  ({params/1e6:.2f}M)")
        print(f"  Size (fp32)  : {size_mb:.2f} MB  (target <= 1.5 MB)")
        print(f"  Layers       : {len(model.layers)}")
        print(f"{'='*60}\n")
        return {'size_mb': size_mb, 'params': params}


# ============================================================================
# GRAD-CAM EXPLAINABILITY
# ============================================================================

class GradCAM:
    """
    Grad-CAM: highlights which leaf areas drove the prediction.
    Red  = high attention (likely infected zone)
    Blue = low attention
    """

    def __init__(self, model):
        self.model = model
        self.last_conv_layer = None
        # Find last convolutional layer automatically
        for layer in reversed(model.layers):
            if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)):
                self.last_conv_layer = layer.name
                break
        if self.last_conv_layer is None:
            raise ValueError("No Conv2D layer found in model.")

        self.grad_model = models.Model(
            inputs  = model.input,
            outputs = [model.get_layer(self.last_conv_layer).output,
                       model.output]
        )

    def compute_heatmap(self, img_array, class_idx=None):
        """
        img_array : float32 array shape (1, H, W, 3) values in [0,1]
        Returns   : heatmap (H, W) in [0, 1]
        """
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_array, training=False)
            if class_idx is None:
                class_idx = tf.argmax(predictions[0])
            class_score = predictions[:, class_idx]

        grads        = tape.gradient(class_score, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap      = tf.squeeze(heatmap).numpy()
        heatmap      = np.maximum(heatmap, 0)
        if heatmap.max() > 0:
            heatmap  = heatmap / heatmap.max()
        return heatmap

    def overlay(self, original_bgr, heatmap, alpha=0.45):
        """Overlay heatmap on original image. Returns BGR image."""
        h, w            = original_bgr.shape[:2]
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_uint8   = np.uint8(255 * heatmap_resized)
        coloured        = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        return cv2.addWeighted(original_bgr, 1 - alpha, coloured, alpha, 0)


# ============================================================================
# COMPLETE PEST DETECTION SYSTEM
# ============================================================================

class PestDetectionSystem:
    """
    Main class used by:
      - Colab notebook  -> training + batch inference + visualisation
      - Raspberry Pi    -> predict_from_image_path()
      - ESP32 server    -> predict_from_array()
    """

    DEFAULT_CLASS_NAMES = [
        'Healthy', 'Fall Armyworm', 'Grasshopper', 'Aphids',
        'Stem Borer', 'Common Rust', 'Gray Leaf Spot',
        'Northern Leaf Blight', 'Leaf Beetle'
    ]

    def __init__(self, model_path=None, num_classes=None, class_names=None):
        self.preprocessor = ImagePreprocessor()
        self.segmenter    = ImageSegmenter()

        # Class names
        if class_names is not None:
            self.class_names = class_names
            self.num_classes = len(class_names)
        else:
            self.class_names = self.DEFAULT_CLASS_NAMES
            self.num_classes = num_classes or len(self.DEFAULT_CLASS_NAMES)

        # Load or build model
        if model_path and os.path.exists(model_path):
            print(f"Loading model from: {model_path}")
            self.model       = keras.models.load_model(model_path)
            self.num_classes = self.model.output_shape[-1]
            print(f"Model loaded  ({self.num_classes} classes)")
        else:
            print("Building new TinyLiteNet model...")
            self.model = TinyLiteNet.build_model(
                input_shape=(224, 224, 3),
                num_classes=self.num_classes
            )

        self.grad_cam = None   # initialised after compile

    # ── Compile ──────────────────────────────────────────────────────
    def compile_model(self, learning_rate=0.001):
        """Compile with metrics required by the Colab notebook."""
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall'),
            ]
        )
        try:
            self.grad_cam = GradCAM(self.model)
        except Exception as e:
            print(f"Grad-CAM init warning: {e}")

    # ── Model info ───────────────────────────────────────────────────
    def get_model_info(self):
        """
        Returns dict with model size and parameters.
        Used by Colab notebook to display model stats.
        """
        params  = self.model.count_params()
        size_mb = (params * 4) / (1024 * 1024)
        return {
            'model_size_mb':       round(size_mb, 2),
            'total_parameters':    params,
            'parameters_millions': round(params / 1e6, 2),
            'num_classes':         self.num_classes,
            'input_shape':         self.model.input_shape,
        }

    # =========================================================================
    # INFERENCE METHODS
    # =========================================================================

    def predict_from_image_path(self, image_path, show_steps=False):
        """
        Load image from disk -> run full 3-step pipeline.
        USE ON: Raspberry Pi
        """
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Cannot load image: {image_path}")
        return self._full_pipeline(image, show_steps=show_steps)

    def predict_from_array(self, image_array):
        """
        Accept raw numpy array (BGR uint8).
        USE ON: ESP32 (image captured -> sent as array) / RPi camera
        Returns lightweight dict: predicted_class + confidence only.
        """
        result = self._full_pipeline(image_array, show_steps=False)
        return {
            'predicted_class': result['predicted_class'],
            'confidence':      result['confidence']
        }

    # Backward-compatible alias
    def predict_single_image_from_array(self, image_array):
        """Alias for predict_from_array() - keeps old code working."""
        return self.predict_from_array(image_array)

    def run_inference_on_batch(self, images_tensor):
        """
        Run model on a tf.data batch tensor.
        USE ON: Colab notebook Step 8.
        images_tensor: float32 (B, H, W, 3) normalised [0,1]
        Returns: predictions array (B, num_classes)
        """
        return self.model.predict(images_tensor, verbose=0)

    # ── Internal pipeline ─────────────────────────────────────────────
    def _full_pipeline(self, image_bgr, show_steps=False):
        """Runs all 3 steps and returns complete result dict."""
        original = image_bgr.copy()

        # STEP 1: Pre-processing
        preprocessed = self.preprocessor.preprocess_image(image_bgr)

        # STEP 2: Segmentation
        seg = self.segmenter.segment_image(preprocessed)

        # STEP 3: CNN Prediction
        model_input   = preprocessed.astype('float32') / 255.0
        model_input   = np.expand_dims(model_input, axis=0)
        predictions   = self.model.predict(model_input, verbose=0)
        predicted_idx = int(np.argmax(predictions[0]))
        confidence    = float(predictions[0][predicted_idx] * 100)

        predicted_class = (self.class_names[predicted_idx]
                           if predicted_idx < len(self.class_names)
                           else f'Class_{predicted_idx}')

        # Grad-CAM
        heatmap = overlay = None
        if self.grad_cam is not None:
            try:
                heatmap = self.grad_cam.compute_heatmap(model_input, predicted_idx)
                overlay = self.grad_cam.overlay(preprocessed, heatmap)
            except Exception:
                pass

        result = {
            'original':        original,
            'preprocessed':    preprocessed,
            'segmented':       seg['segmented'],
            'masked':          seg['masked'],
            'binary':          seg['binary'],
            'edges':           seg['edges'],
            'predicted_class': predicted_class,
            'predicted_idx':   predicted_idx,
            'confidence':      confidence,
            'all_probs':       predictions[0],
            'heatmap':         heatmap,
            'overlay':         overlay,
        }

        if show_steps:
            self._visualise_steps(result)

        return result

    # =========================================================================
    # VISUALISATION METHODS  (Colab-friendly, display inline)
    # =========================================================================

    def _visualise_steps(self, result):
        """Show all 3 processing steps in one figure."""
        titles = ['1. Original', '2. Pre-processed', '3. K-means Seg.',
                  '4. Diseased Mask', '5. Otsu Binary', '6. Grad-CAM']
        images = [
            result['original'],
            result['preprocessed'],
            result['segmented'],
            result['masked'],
            result['binary'],
            result['overlay'] if result['overlay'] is not None else result['preprocessed']
        ]

        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        axes = axes.flatten()

        for i, (title, img) in enumerate(zip(titles, images)):
            if img is None:
                axes[i].text(0.5, 0.5, 'N/A', ha='center', va='center')
            elif len(img.shape) == 2:
                axes[i].imshow(img, cmap='gray')
            else:
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            if i == 5:
                title = (f"Prediction: {result['predicted_class']}\n"
                         f"Confidence: {result['confidence']:.1f}%")

            axes[i].set_title(title, fontsize=11, fontweight='bold')
            axes[i].axis('off')

        fig.suptitle('Pest Detection - All 3 Processing Steps',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def show_predictions(self, images_tensor, labels_tensor, class_names=None):
        """
        Show 9 sample predictions from a Colab val_ds batch.
        USE ON: Colab notebook Step 8.

        images_tensor : float32 tensor (B, H, W, 3) normalised [0,1]
        labels_tensor : one-hot tensor (B, num_classes)
        class_names   : list of class name strings (optional override)
        """
        if class_names is not None:
            self.class_names = class_names

        predictions = self.run_inference_on_batch(images_tensor)
        n           = min(9, len(images_tensor))

        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        axes = axes.flatten()

        for i in range(n):
            img        = images_tensor[i].numpy()
            true_idx   = int(np.argmax(labels_tensor[i]))
            pred_idx   = int(np.argmax(predictions[i]))
            confidence = float(predictions[i][pred_idx] * 100)

            true_name = (self.class_names[true_idx]
                         if true_idx < len(self.class_names) else f'Class_{true_idx}')
            pred_name = (self.class_names[pred_idx]
                         if pred_idx < len(self.class_names) else f'Class_{pred_idx}')

            axes[i].imshow(img)
            color = 'green' if true_idx == pred_idx else 'red'
            axes[i].set_title(
                f"True: {true_name}\nPred: {pred_name}\n({confidence:.1f}%)",
                color=color, fontsize=9, fontweight='bold'
            )
            axes[i].axis('off')

        for j in range(n, 9):
            axes[j].axis('off')

        correct   = mpatches.Patch(color='green', label='Correct prediction')
        incorrect = mpatches.Patch(color='red',   label='Wrong prediction')
        fig.legend(handles=[correct, incorrect],
                   loc='lower center', ncol=2, fontsize=11)

        fig.suptitle('Sample Predictions - Pest Detection',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        plt.show()

    def plot_training_history(self, history):
        """
        Plot accuracy and loss curves after training.
        USE ON: Colab notebook Step 6.
        """
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Accuracy
        axes[0].plot(history.history['accuracy'],     label='Train',      linewidth=2)
        axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0].axhline(y=0.986, color='red', linestyle='--', label='Target 98.6%')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title('Model Accuracy - TinyLiteNet', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Loss
        axes[1].plot(history.history['loss'],     label='Train',      linewidth=2)
        axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Model Loss - TinyLiteNet', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved: training_history.png")

    def show_confusion_matrix(self, val_ds, class_names=None):
        """
        Generate and display confusion matrix from a tf.data dataset.
        USE ON: Colab after training to evaluate model.
        """
        try:
            import seaborn as sns
        except ImportError:
            print("seaborn not installed. Run: pip install seaborn")
            return

        from sklearn.metrics import confusion_matrix, classification_report

        if class_names is not None:
            self.class_names = class_names

        y_true, y_pred = [], []
        for images, labels in val_ds:
            preds  = self.run_inference_on_batch(images)
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            y_pred.extend(np.argmax(preds, axis=1))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.class_names,
                    yticklabels=self.class_names)
        plt.xlabel('Predicted', fontsize=12)
        plt.ylabel('True',      fontsize=12)
        plt.title('Confusion Matrix - Pest & Disease Detection',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved: confusion_matrix.png")
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=self.class_names))


# ============================================================================
# QUICK SELF-TEST
# ============================================================================

if __name__ == '__main__':
    print("=" * 65)
    print("  IMPROVED PEST DETECTION SYSTEM - SELF TEST")
    print("=" * 65)

    system = PestDetectionSystem(num_classes=9)
    system.compile_model()

    info = system.get_model_info()
    print(f"\n  Model size   : {info['model_size_mb']} MB   (target <= 1.5 MB)")
    print(f"  Parameters   : {info['parameters_millions']}M   (target <= 2M)")

    dummy_bgr    = np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8)
    preprocessed = ImagePreprocessor.preprocess_image(dummy_bgr)
    print(f"\n  Pre-processing : {dummy_bgr.shape} -> {preprocessed.shape}  OK")

    seg = ImageSegmenter.segment_image(preprocessed)
    print(f"  Segmentation   : keys = {list(seg.keys())}  OK")

    result = system.predict_from_array(dummy_bgr)
    print(f"\n  Inference test :")
    print(f"    Predicted class : {result['predicted_class']}")
    print(f"    Confidence      : {result['confidence']:.1f}%")

    print("\n" + "=" * 65)
    print("  All tests passed - system is ready!")
    print("=" * 65)

  IMPROVED PEST DETECTION SYSTEM - SELF TEST
Building new TinyLiteNet model...


I0000 00:00:1773168071.358464      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773168071.361688      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



  Model size   : 1.76 MB   (target <= 1.5 MB)
  Parameters   : 0.46M   (target <= 2M)

  Pre-processing : (300, 300, 3) -> (224, 224, 3)  OK
  Segmentation   : keys = ['original', 'segmented', 'masked', 'binary', 'edges', 'mask']  OK


I0000 00:00:1773168073.949131     129 service.cc:152] XLA service 0x7db568006010 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773168073.949171     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773168073.949177     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773168074.299560     129 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773168080.108114     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



  Inference test :
    Predicted class : Healthy
    Confidence      : 11.1%

  All tests passed - system is ready!


In [3]:
# Create the file using raw string format
content = r'''import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# STEP 1: ENHANCED PRE-PROCESSING  (From Paper 2 - Khirade & Patil 2015)
# ============================================================================

class ImagePreprocessor:
    """
    Complete pre-processing pipeline from Paper 2.

    Steps applied in order:
      1. Resize to target_size
      2. Gaussian blur      -> removes camera noise
      3. Histogram equaliz  -> improves contrast in any lighting
      4. Contrast enhance   -> makes disease features clearer
    """

    @staticmethod
    def noise_removal(image, kernel_size=5):
        """Gaussian blur to remove noise"""
        return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

    @staticmethod
    def histogram_equalization(image):
        """
        Histogram equalization on Y channel (YUV space).
        Paper 2: 'distributes the intensities of the images'
        Keeps colour, only changes brightness distribution.
        """
        img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
        img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
        return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

    @staticmethod
    def enhance_contrast(image, alpha=1.3, beta=0):
        """
        Contrast enhancement.
        alpha > 1 increases contrast.
        """
        return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    @staticmethod
    def preprocess_image(image, target_size=(224, 224)):
        """
        Full pipeline called for every image before CNN.
        Works on a numpy BGR array (same as cv2.imread output).
        """
        image = cv2.resize(image, target_size)
        image = ImagePreprocessor.noise_removal(image)
        image = ImagePreprocessor.histogram_equalization(image)
        image = ImagePreprocessor.enhance_contrast(image)
        return image


# ============================================================================
# STEP 2: SEGMENTATION LAYER  (From Paper 2 - Khirade & Patil 2015)
# ============================================================================

class ImageSegmenter:
    """
    Implements three segmentation methods from Paper 2:
      1. K-means clustering (k=3)  -> separates healthy/diseased/background
      2. Green pixel masking        -> removes healthy green tissue
      3. Otsu thresholding          -> binary mask of diseased region
      + Canny edge detection        -> highlights lesion boundaries
    """

    @staticmethod
    def kmeans_segmentation(image, k=3):
        """
        K-means clustering.
        Paper 2: 'K-means clustering is more accurate than other methods'
        k=3 -> healthy tissue, diseased tissue, background
        """
        pixel_values = np.float32(image.reshape((-1, 3)))
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
        _, labels, centers = cv2.kmeans(
            pixel_values, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
        )
        centers  = np.uint8(centers)
        segmented = centers[labels.flatten()].reshape(image.shape)
        return segmented

    @staticmethod
    def green_pixel_masking(image):
        """
        Remove healthy (green) pixels so CNN focuses on diseased areas.
        Paper 2: 'green pixels masked and removed'
        """
        hsv           = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        lower_green   = np.array([35, 40, 40])
        upper_green   = np.array([85, 255, 255])
        green_mask    = cv2.inRange(hsv, lower_green, upper_green)
        diseased_mask = cv2.bitwise_not(green_mask)
        masked        = cv2.bitwise_and(image, image, mask=diseased_mask)
        return masked, diseased_mask

    @staticmethod
    def otsu_thresholding(image):
        """
        Otsu automatic thresholding.
        Paper 2: 'creates binary images from grey-level images'
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary

    @staticmethod
    def segment_image(image):
        """
        Full segmentation pipeline.
        Returns dict so every stage can be visualised separately.
        """
        segmented          = ImageSegmenter.kmeans_segmentation(image, k=3)
        masked, mask       = ImageSegmenter.green_pixel_masking(segmented)
        binary             = ImageSegmenter.otsu_thresholding(masked)
        gray               = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
        edges              = cv2.Canny(gray, 50, 150)

        return {
            'original':  image,
            'segmented': segmented,
            'masked':    masked,
            'binary':    binary,
            'edges':     edges,
            'mask':      mask
        }


# ============================================================================
# STEP 3: TINY-LITENET CNN  (From Paper 1 - Nyakuri et al. 2025)
# ============================================================================

class TinyLiteNet:
    """
    Lightweight CNN model.
    Paper 1 specs: 1.2 MB, 1.48 M parameters, 16 ms inference on RPi5.
    Architecture: MobileNetV2 inspired + 6 Squeeze-Excitation (SE) blocks.
    """

    @staticmethod
    def squeeze_excitation_block(x, ratio=16):
        """
        SE block - recalibrates channel-wise feature responses.
        Paper 1: '6 SE depthwise blocks to enhance feature representation'
        """
        channels = x.shape[-1]
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Dense(max(1, channels // ratio), activation='relu')(se)
        se = layers.Dense(channels, activation='sigmoid')(se)
        se = layers.Reshape((1, 1, channels))(se)
        return layers.Multiply()([x, se])

    @staticmethod
    def depthwise_block(x, filters, stride=1):
        """Depthwise-separable conv block (MobileNetV2 style)."""
        x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)
        x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)
        return x

    @staticmethod
    def build_model(input_shape=(224, 224, 3), num_classes=9):
        """
        Build Tiny-LiteNet.
        Target: ~1.2 MB, ~1.48 M parameters.
        """
        inputs = layers.Input(shape=input_shape, name='input_image')

        # Stem conv
        x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)

        # 6 Depthwise + SE blocks  (Paper 1: 6 SE depthwise blocks)
        configs = [(64,1), (128,2), (128,1), (256,2), (256,1), (512,2)]
        for filters, stride in configs:
            x = TinyLiteNet.depthwise_block(x, filters, stride)
            x = TinyLiteNet.squeeze_excitation_block(x)

        # Classification head
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

        model = models.Model(inputs, outputs, name='TinyLiteNet')
        return model

    @staticmethod
    def model_info(model):
        """Print model size and parameter count."""
        params  = model.count_params()
        size_mb = (params * 4) / (1024 * 1024)
        print(f"\n{'='*60}")
        print(f"  Model        : {model.name}")
        print(f"  Parameters   : {params:,}  ({params/1e6:.2f}M)")
        print(f"  Size (fp32)  : {size_mb:.2f} MB  (target <= 1.5 MB)")
        print(f"  Layers       : {len(model.layers)}")
        print(f"{'='*60}\n")
        return {'size_mb': size_mb, 'params': params}


# ============================================================================
# GRAD-CAM EXPLAINABILITY
# ============================================================================

class GradCAM:
    """
    Grad-CAM: highlights which leaf areas drove the prediction.
    Red  = high attention (likely infected zone)
    Blue = low attention
    """

    def __init__(self, model):
        self.model = model
        self.last_conv_layer = None
        # Find last convolutional layer automatically
        for layer in reversed(model.layers):
            if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)):
                self.last_conv_layer = layer.name
                break
        if self.last_conv_layer is None:
            raise ValueError("No Conv2D layer found in model.")

        self.grad_model = models.Model(
            inputs  = model.input,
            outputs = [model.get_layer(self.last_conv_layer).output,
                       model.output]
        )

    def compute_heatmap(self, img_array, class_idx=None):
        """
        img_array : float32 array shape (1, H, W, 3) values in [0,1]
        Returns   : heatmap (H, W) in [0, 1]
        """
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_array, training=False)
            if class_idx is None:
                class_idx = tf.argmax(predictions[0])
            class_score = predictions[:, class_idx]

        grads        = tape.gradient(class_score, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap      = tf.squeeze(heatmap).numpy()
        heatmap      = np.maximum(heatmap, 0)
        if heatmap.max() > 0:
            heatmap  = heatmap / heatmap.max()
        return heatmap

    def overlay(self, original_bgr, heatmap, alpha=0.45):
        """Overlay heatmap on original image. Returns BGR image."""
        h, w            = original_bgr.shape[:2]
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_uint8   = np.uint8(255 * heatmap_resized)
        coloured        = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        return cv2.addWeighted(original_bgr, 1 - alpha, coloured, alpha, 0)


# ============================================================================
# COMPLETE PEST DETECTION SYSTEM
# ============================================================================

class PestDetectionSystem:
    """
    Main class used by:
      - Colab notebook  -> training + batch inference + visualisation
      - Raspberry Pi    -> predict_from_image_path()
      - ESP32 server    -> predict_from_array()
    """

    DEFAULT_CLASS_NAMES = [
        'Healthy', 'Fall Armyworm', 'Grasshopper', 'Aphids',
        'Stem Borer', 'Common Rust', 'Gray Leaf Spot',
        'Northern Leaf Blight', 'Leaf Beetle'
    ]

    def __init__(self, model_path=None, num_classes=None, class_names=None):
        self.preprocessor = ImagePreprocessor()
        self.segmenter    = ImageSegmenter()

        # Class names
        if class_names is not None:
            self.class_names = class_names
            self.num_classes = len(class_names)
        else:
            self.class_names = self.DEFAULT_CLASS_NAMES
            self.num_classes = num_classes or len(self.DEFAULT_CLASS_NAMES)

        # Load or build model
        if model_path and os.path.exists(model_path):
            print(f"Loading model from: {model_path}")
            self.model       = keras.models.load_model(model_path)
            self.num_classes = self.model.output_shape[-1]
            print(f"Model loaded  ({self.num_classes} classes)")
        else:
            print("Building new TinyLiteNet model...")
            self.model = TinyLiteNet.build_model(
                input_shape=(224, 224, 3),
                num_classes=self.num_classes
            )

        self.grad_cam = None   # initialised after compile

    # ── Compile ──────────────────────────────────────────────────────
    def compile_model(self, learning_rate=0.001):
        """Compile with metrics required by the Colab notebook."""
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall'),
            ]
        )
        try:
            self.grad_cam = GradCAM(self.model)
        except Exception as e:
            print(f"Grad-CAM init warning: {e}")

    # ── Model info ───────────────────────────────────────────────────
    def get_model_info(self):
        """
        Returns dict with model size and parameters.
        Used by Colab notebook to display model stats.
        """
        params  = self.model.count_params()
        size_mb = (params * 4) / (1024 * 1024)
        return {
            'model_size_mb':       round(size_mb, 2),
            'total_parameters':    params,
            'parameters_millions': round(params / 1e6, 2),
            'num_classes':         self.num_classes,
            'input_shape':         self.model.input_shape,
        }

    # =========================================================================
    # INFERENCE METHODS
    # =========================================================================

    def predict_from_image_path(self, image_path, show_steps=False):
        """
        Load image from disk -> run full 3-step pipeline.
        USE ON: Raspberry Pi
        """
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Cannot load image: {image_path}")
        return self._full_pipeline(image, show_steps=show_steps)

    def predict_from_array(self, image_array):
        """
        Accept raw numpy array (BGR uint8).
        USE ON: ESP32 (image captured -> sent as array) / RPi camera
        Returns lightweight dict: predicted_class + confidence only.
        """
        result = self._full_pipeline(image_array, show_steps=False)
        return {
            'predicted_class': result['predicted_class'],
            'confidence':      result['confidence']
        }

    # Backward-compatible alias
    def predict_single_image_from_array(self, image_array):
        """Alias for predict_from_array() - keeps old code working."""
        return self.predict_from_array(image_array)

    def run_inference_on_batch(self, images_tensor):
        """
        Run model on a tf.data batch tensor.
        USE ON: Colab notebook Step 8.
        images_tensor: float32 (B, H, W, 3) normalised [0,1]
        Returns: predictions array (B, num_classes)
        """
        return self.model.predict(images_tensor, verbose=0)

    # ── Internal pipeline ─────────────────────────────────────────────
    def _full_pipeline(self, image_bgr, show_steps=False):
        """Runs all 3 steps and returns complete result dict."""
        original = image_bgr.copy()

        # STEP 1: Pre-processing
        preprocessed = self.preprocessor.preprocess_image(image_bgr)

        # STEP 2: Segmentation
        seg = self.segmenter.segment_image(preprocessed)

        # STEP 3: CNN Prediction
        model_input   = preprocessed.astype('float32') / 255.0
        model_input   = np.expand_dims(model_input, axis=0)
        predictions   = self.model.predict(model_input, verbose=0)
        predicted_idx = int(np.argmax(predictions[0]))
        confidence    = float(predictions[0][predicted_idx] * 100)

        predicted_class = (self.class_names[predicted_idx]
                           if predicted_idx < len(self.class_names)
                           else f'Class_{predicted_idx}')

        # Grad-CAM
        heatmap = overlay = None
        if self.grad_cam is not None:
            try:
                heatmap = self.grad_cam.compute_heatmap(model_input, predicted_idx)
                overlay = self.grad_cam.overlay(preprocessed, heatmap)
            except Exception:
                pass

        result = {
            'original':        original,
            'preprocessed':    preprocessed,
            'segmented':       seg['segmented'],
            'masked':          seg['masked'],
            'binary':          seg['binary'],
            'edges':           seg['edges'],
            'predicted_class': predicted_class,
            'predicted_idx':   predicted_idx,
            'confidence':      confidence,
            'all_probs':       predictions[0],
            'heatmap':         heatmap,
            'overlay':         overlay,
        }

        if show_steps:
            self._visualise_steps(result)

        return result

    # =========================================================================
    # VISUALISATION METHODS  (Colab-friendly, display inline)
    # =========================================================================

    def _visualise_steps(self, result):
        """Show all 3 processing steps in one figure."""
        titles = ['1. Original', '2. Pre-processed', '3. K-means Seg.',
                  '4. Diseased Mask', '5. Otsu Binary', '6. Grad-CAM']
        images = [
            result['original'],
            result['preprocessed'],
            result['segmented'],
            result['masked'],
            result['binary'],
            result['overlay'] if result['overlay'] is not None else result['preprocessed']
        ]

        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        axes = axes.flatten()

        for i, (title, img) in enumerate(zip(titles, images)):
            if img is None:
                axes[i].text(0.5, 0.5, 'N/A', ha='center', va='center')
            elif len(img.shape) == 2:
                axes[i].imshow(img, cmap='gray')
            else:
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            if i == 5:
                title = (f"Prediction: {result['predicted_class']}\n"
                         f"Confidence: {result['confidence']:.1f}%")

            axes[i].set_title(title, fontsize=11, fontweight='bold')
            axes[i].axis('off')

        fig.suptitle('Pest Detection - All 3 Processing Steps',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def show_predictions(self, images_tensor, labels_tensor, class_names=None):
        """
        Show 9 sample predictions from a Colab val_ds batch.
        USE ON: Colab notebook Step 8.

        images_tensor : float32 tensor (B, H, W, 3) normalised [0,1]
        labels_tensor : one-hot tensor (B, num_classes)
        class_names   : list of class name strings (optional override)
        """
        if class_names is not None:
            self.class_names = class_names

        predictions = self.run_inference_on_batch(images_tensor)
        n           = min(9, len(images_tensor))

        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        axes = axes.flatten()

        for i in range(n):
            img        = images_tensor[i].numpy()
            true_idx   = int(np.argmax(labels_tensor[i]))
            pred_idx   = int(np.argmax(predictions[i]))
            confidence = float(predictions[i][pred_idx] * 100)

            true_name = (self.class_names[true_idx]
                         if true_idx < len(self.class_names) else f'Class_{true_idx}')
            pred_name = (self.class_names[pred_idx]
                         if pred_idx < len(self.class_names) else f'Class_{pred_idx}')

            axes[i].imshow(img)
            color = 'green' if true_idx == pred_idx else 'red'
            axes[i].set_title(
                f"True: {true_name}\nPred: {pred_name}\n({confidence:.1f}%)",
                color=color, fontsize=9, fontweight='bold'
            )
            axes[i].axis('off')

        for j in range(n, 9):
            axes[j].axis('off')

        correct   = mpatches.Patch(color='green', label='Correct prediction')
        incorrect = mpatches.Patch(color='red',   label='Wrong prediction')
        fig.legend(handles=[correct, incorrect],
                   loc='lower center', ncol=2, fontsize=11)

        fig.suptitle('Sample Predictions - Pest Detection',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        plt.show()

    def plot_training_history(self, history):
        """
        Plot accuracy and loss curves after training.
        USE ON: Colab notebook Step 6.
        """
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Accuracy
        axes[0].plot(history.history['accuracy'],     label='Train',      linewidth=2)
        axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0].axhline(y=0.986, color='red', linestyle='--', label='Target 98.6%')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title('Model Accuracy - TinyLiteNet', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Loss
        axes[1].plot(history.history['loss'],     label='Train',      linewidth=2)
        axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Model Loss - TinyLiteNet', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved: training_history.png")

    def show_confusion_matrix(self, val_ds, class_names=None):
        """
        Generate and display confusion matrix from a tf.data dataset.
        USE ON: Colab after training to evaluate model.
        """
        try:
            import seaborn as sns
        except ImportError:
            print("seaborn not installed. Run: pip install seaborn")
            return

        from sklearn.metrics import confusion_matrix, classification_report

        if class_names is not None:
            self.class_names = class_names

        y_true, y_pred = [], []
        for images, labels in val_ds:
            preds  = self.run_inference_on_batch(images)
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            y_pred.extend(np.argmax(preds, axis=1))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.class_names,
                    yticklabels=self.class_names)
        plt.xlabel('Predicted', fontsize=12)
        plt.ylabel('True',      fontsize=12)
        plt.title('Confusion Matrix - Pest & Disease Detection',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved: confusion_matrix.png")
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=self.class_names))


# ============================================================================
# QUICK SELF-TEST
# ============================================================================

if __name__ == '__main__':
    print("=" * 65)
    print("  IMPROVED PEST DETECTION SYSTEM - SELF TEST")
    print("=" * 65)

    system = PestDetectionSystem(num_classes=9)
    system.compile_model()

    info = system.get_model_info()
    print(f"\n  Model size   : {info['model_size_mb']} MB   (target <= 1.5 MB)")
    print(f"  Parameters   : {info['parameters_millions']}M   (target <= 2M)")

    dummy_bgr    = np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8)
    preprocessed = ImagePreprocessor.preprocess_image(dummy_bgr)
    print(f"\n  Pre-processing : {dummy_bgr.shape} -> {preprocessed.shape}  OK")

    seg = ImageSegmenter.segment_image(preprocessed)
    print(f"  Segmentation   : keys = {list(seg.keys())}  OK")

    result = system.predict_from_array(dummy_bgr)
    print(f"\n  Inference test :")
    print(f"    Predicted class : {result['predicted_class']}")
    print(f"    Confidence      : {result['confidence']:.1f}%")

    print("\n" + "=" * 65)
    print("  All tests passed - system is ready!")
    print("=" * 65)
'''

# Write the content to file
with open('improved_pest_detection.py', 'w') as f:
    f.write(content)

print("✅ File created successfully with raw string!")

# Verify the file
print("\nFile exists:", os.path.exists('improved_pest_detection.py'))
print("File size:", os.path.getsize('improved_pest_detection.py'), "bytes")

✅ File created successfully with raw string!

File exists: True
File size: 25036 bytes


In [ ]:
"""
train_local_laptop.py
=====================
Train the Pest Detection (Tiny-LiteNet) model on YOUR LAPTOP.
No Colab, no internet needed after dataset is ready.

Based on Research Papers:
  Paper 1: Nyakuri et al. (2025) - AI and IoT-powered edge device
  Paper 2: Khirade & Patil (2015) - Plant Disease Detection Using Image Processing

HOW TO USE THIS FILE:
  Step 1 — Install requirements (run once):
            pip install tensorflow opencv-python scikit-learn matplotlib seaborn kaggle

  Step 2 — Download dataset (choose ONE option below):
            Option A — Kaggle auto-download  -> set USE_KAGGLE = True
            Option B — Manual folder         -> set DATASET_PATH to your folder

  Step 3 — Run:
            python train_local_laptop.py

  Step 4 — Get trained model:
            trained_models/tinylitenet_best.keras     <- best checkpoint
            trained_models/tinylitenet_final.keras    <- final saved model
            trained_models/class_names.json           <- class labels

FOLDER STRUCTURE EXPECTED (if manual):
  dataset/
  ├── Healthy/
  ├── Fall_Armyworm/
  ├── Grasshopper/
  ├── Aphids/
  └── ...one sub-folder per class

MINIMUM LAPTOP SPECS:
  RAM  : 8 GB  (16 GB recommended)
  CPU  : Any modern CPU (Intel i5 / Ryzen 5 or better)
  Disk : 5 GB free
  Time : 1–3 hours depending on dataset size and CPU speed
  GPU  : Optional but makes training 5–10x faster
"""

# ============================================================
# USER SETTINGS  ← CHANGE THESE
# ============================================================

# --- Dataset ---
USE_KAGGLE     = False  # Set to True if you want to download from Kaggle
KAGGLE_DATASET = 'emmarex/plantdisease'  # Kaggle dataset name
DATASET_PATH   = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'    
                                                         # Also used as download destination

# --- Training settings ---
IMAGE_SIZE   = (224, 224)   # Must match model input (do not change)
BATCH_SIZE   = 8           # Lower if you get out-of-memory error (try 8)
EPOCHS       = 50         # Max epochs; early stopping will stop earlier
LEARNING_RATE = 0.001

# --- Output folder ---
# In Kaggle notebook, use working directory
OUTPUT_DIR = '/kaggle/working/trained_models'

# ============================================================
# END OF USER SETTINGS
# ============================================================


import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

# ── 1. Check required packages ────────────────────────────────────────────────
def check_packages():
    required = {
        'tensorflow': 'tensorflow',
        'cv2':        'opencv-python',
        'sklearn':    'scikit-learn',
        'matplotlib': 'matplotlib',
        'numpy':      'numpy',
    }
    missing = []
    for module, pip_name in required.items():
        try:
            __import__(module)
        except ImportError:
            missing.append(pip_name)

    if missing:
        print("\n❌ Missing packages. Install them first:\n")
        print(f"   pip install {' '.join(missing)}\n")
        sys.exit(1)

    print("✅ All required packages found.")

check_packages()


# ── 2. Now safe to import ─────────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Check if improved_pest_detection.py is in the same folder
if not os.path.exists('improved_pest_detection.py'):
    print("\n❌  improved_pest_detection.py NOT found in current folder.")
    print("   Creating it now...")
    
    # Instead of exiting, we'll create the file
    with open('improved_pest_detection.py', 'w') as f:
        f.write('''import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# STEP 1: ENHANCED PRE-PROCESSING  (From Paper 2 - Khirade & Patil 2015)
# ============================================================================

class ImagePreprocessor:
    """
    Complete pre-processing pipeline from Paper 2.

    Steps applied in order:
      1. Resize to target_size
      2. Gaussian blur      -> removes camera noise
      3. Histogram equaliz  -> improves contrast in any lighting
      4. Contrast enhance   -> makes disease features clearer
    """

    @staticmethod
    def noise_removal(image, kernel_size=5):
        """Gaussian blur to remove noise"""
        return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

    @staticmethod
    def histogram_equalization(image):
        """
        Histogram equalization on Y channel (YUV space).
        Paper 2: 'distributes the intensities of the images'
        Keeps colour, only changes brightness distribution.
        """
        img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
        img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
        return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

    @staticmethod
    def enhance_contrast(image, alpha=1.3, beta=0):
        """
        Contrast enhancement.
        alpha > 1 increases contrast.
        """
        return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    @staticmethod
    def preprocess_image(image, target_size=(224, 224)):
        """
        Full pipeline called for every image before CNN.
        Works on a numpy BGR array (same as cv2.imread output).
        """
        image = cv2.resize(image, target_size)
        image = ImagePreprocessor.noise_removal(image)
        image = ImagePreprocessor.histogram_equalization(image)
        image = ImagePreprocessor.enhance_contrast(image)
        return image


# ============================================================================
# STEP 2: SEGMENTATION LAYER  (From Paper 2 - Khirade & Patil 2015)
# ============================================================================

class ImageSegmenter:
    """
    Implements three segmentation methods from Paper 2:
      1. K-means clustering (k=3)  -> separates healthy/diseased/background
      2. Green pixel masking        -> removes healthy green tissue
      3. Otsu thresholding          -> binary mask of diseased region
      + Canny edge detection        -> highlights lesion boundaries
    """

    @staticmethod
    def kmeans_segmentation(image, k=3):
        """
        K-means clustering.
        Paper 2: 'K-means clustering is more accurate than other methods'
        k=3 -> healthy tissue, diseased tissue, background
        """
        pixel_values = np.float32(image.reshape((-1, 3)))
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
        _, labels, centers = cv2.kmeans(
            pixel_values, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
        )
        centers  = np.uint8(centers)
        segmented = centers[labels.flatten()].reshape(image.shape)
        return segmented

    @staticmethod
    def green_pixel_masking(image):
        """
        Remove healthy (green) pixels so CNN focuses on diseased areas.
        Paper 2: 'green pixels masked and removed'
        """
        hsv           = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        lower_green   = np.array([35, 40, 40])
        upper_green   = np.array([85, 255, 255])
        green_mask    = cv2.inRange(hsv, lower_green, upper_green)
        diseased_mask = cv2.bitwise_not(green_mask)
        masked        = cv2.bitwise_and(image, image, mask=diseased_mask)
        return masked, diseased_mask

    @staticmethod
    def otsu_thresholding(image):
        """
        Otsu automatic thresholding.
        Paper 2: 'creates binary images from grey-level images'
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary

    @staticmethod
    def segment_image(image):
        """
        Full segmentation pipeline.
        Returns dict so every stage can be visualised separately.
        """
        segmented          = ImageSegmenter.kmeans_segmentation(image, k=3)
        masked, mask       = ImageSegmenter.green_pixel_masking(segmented)
        binary             = ImageSegmenter.otsu_thresholding(masked)
        gray               = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
        edges              = cv2.Canny(gray, 50, 150)

        return {
            'original':  image,
            'segmented': segmented,
            'masked':    masked,
            'binary':    binary,
            'edges':     edges,
            'mask':      mask
        }


# ============================================================================
# STEP 3: TINY-LITENET CNN  (From Paper 1 - Nyakuri et al. 2025)
# ============================================================================

class TinyLiteNet:
    """
    Lightweight CNN model.
    Paper 1 specs: 1.2 MB, 1.48 M parameters, 16 ms inference on RPi5.
    Architecture: MobileNetV2 inspired + 6 Squeeze-Excitation (SE) blocks.
    """

    @staticmethod
    def squeeze_excitation_block(x, ratio=16):
        """
        SE block - recalibrates channel-wise feature responses.
        Paper 1: '6 SE depthwise blocks to enhance feature representation'
        """
        channels = x.shape[-1]
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Dense(max(1, channels // ratio), activation='relu')(se)
        se = layers.Dense(channels, activation='sigmoid')(se)
        se = layers.Reshape((1, 1, channels))(se)
        return layers.Multiply()([x, se])

    @staticmethod
    def depthwise_block(x, filters, stride=1):
        """Depthwise-separable conv block (MobileNetV2 style)."""
        x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)
        x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)
        return x

    @staticmethod
    def build_model(input_shape=(224, 224, 3), num_classes=9):
        """
        Build Tiny-LiteNet.
        Target: ~1.2 MB, ~1.48 M parameters.
        """
        inputs = layers.Input(shape=input_shape, name='input_image')

        # Stem conv
        x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6)(x)

        # 6 Depthwise + SE blocks  (Paper 1: 6 SE depthwise blocks)
        configs = [(64,1), (128,2), (128,1), (256,2), (256,1), (512,2)]
        for filters, stride in configs:
            x = TinyLiteNet.depthwise_block(x, filters, stride)
            x = TinyLiteNet.squeeze_excitation_block(x)

        # Classification head
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

        model = models.Model(inputs, outputs, name='TinyLiteNet')
        return model

    @staticmethod
    def model_info(model):
        """Print model size and parameter count."""
        params  = model.count_params()
        size_mb = (params * 4) / (1024 * 1024)
        print(f"\\n{'='*60}")
        print(f"  Model        : {model.name}")
        print(f"  Parameters   : {params:,}  ({params/1e6:.2f}M)")
        print(f"  Size (fp32)  : {size_mb:.2f} MB  (target <= 1.5 MB)")
        print(f"  Layers       : {len(model.layers)}")
        print(f"{'='*60}\\n")
        return {'size_mb': size_mb, 'params': params}


# ============================================================================
# GRAD-CAM EXPLAINABILITY
# ============================================================================

class GradCAM:
    """
    Grad-CAM: highlights which leaf areas drove the prediction.
    Red  = high attention (likely infected zone)
    Blue = low attention
    """

    def __init__(self, model):
        self.model = model
        self.last_conv_layer = None
        # Find last convolutional layer automatically
        for layer in reversed(model.layers):
            if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)):
                self.last_conv_layer = layer.name
                break
        if self.last_conv_layer is None:
            raise ValueError("No Conv2D layer found in model.")

        self.grad_model = models.Model(
            inputs  = model.input,
            outputs = [model.get_layer(self.last_conv_layer).output,
                       model.output]
        )

    def compute_heatmap(self, img_array, class_idx=None):
        """
        img_array : float32 array shape (1, H, W, 3) values in [0,1]
        Returns   : heatmap (H, W) in [0, 1]
        """
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_array, training=False)
            if class_idx is None:
                class_idx = tf.argmax(predictions[0])
            class_score = predictions[:, class_idx]

        grads        = tape.gradient(class_score, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap      = tf.squeeze(heatmap).numpy()
        heatmap      = np.maximum(heatmap, 0)
        if heatmap.max() > 0:
            heatmap  = heatmap / heatmap.max()
        return heatmap

    def overlay(self, original_bgr, heatmap, alpha=0.45):
        """Overlay heatmap on original image. Returns BGR image."""
        h, w            = original_bgr.shape[:2]
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_uint8   = np.uint8(255 * heatmap_resized)
        coloured        = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        return cv2.addWeighted(original_bgr, 1 - alpha, coloured, alpha, 0)


# ============================================================================
# COMPLETE PEST DETECTION SYSTEM
# ============================================================================

class PestDetectionSystem:
    """
    Main class used by:
      - Colab notebook  -> training + batch inference + visualisation
      - Raspberry Pi    -> predict_from_image_path()
      - ESP32 server    -> predict_from_array()
    """

    DEFAULT_CLASS_NAMES = [
        'Healthy', 'Fall Armyworm', 'Grasshopper', 'Aphids',
        'Stem Borer', 'Common Rust', 'Gray Leaf Spot',
        'Northern Leaf Blight', 'Leaf Beetle'
    ]

    def __init__(self, model_path=None, num_classes=None, class_names=None):
        self.preprocessor = ImagePreprocessor()
        self.segmenter    = ImageSegmenter()

        # Class names
        if class_names is not None:
            self.class_names = class_names
            self.num_classes = len(class_names)
        else:
            self.class_names = self.DEFAULT_CLASS_NAMES
            self.num_classes = num_classes or len(self.DEFAULT_CLASS_NAMES)

        # Load or build model
        if model_path and os.path.exists(model_path):
            print(f"Loading model from: {model_path}")
            self.model       = keras.models.load_model(model_path)
            self.num_classes = self.model.output_shape[-1]
            print(f"Model loaded  ({self.num_classes} classes)")
        else:
            print("Building new TinyLiteNet model...")
            self.model = TinyLiteNet.build_model(
                input_shape=(224, 224, 3),
                num_classes=self.num_classes
            )

        self.grad_cam = None   # initialised after compile

    # ── Compile ──────────────────────────────────────────────────────
    def compile_model(self, learning_rate=0.001):
        """Compile with metrics required by the Colab notebook."""
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall'),
            ]
        )
        try:
            self.grad_cam = GradCAM(self.model)
        except Exception as e:
            print(f"Grad-CAM init warning: {e}")

    # ── Model info ───────────────────────────────────────────────────
    def get_model_info(self):
        """
        Returns dict with model size and parameters.
        Used by Colab notebook to display model stats.
        """
        params  = self.model.count_params()
        size_mb = (params * 4) / (1024 * 1024)
        return {
            'model_size_mb':       round(size_mb, 2),
            'total_parameters':    params,
            'parameters_millions': round(params / 1e6, 2),
            'num_classes':         self.num_classes,
            'input_shape':         self.model.input_shape,
        }

    # =========================================================================
    # INFERENCE METHODS
    # =========================================================================

    def predict_from_image_path(self, image_path, show_steps=False):
        """
        Load image from disk -> run full 3-step pipeline.
        USE ON: Raspberry Pi
        """
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Cannot load image: {image_path}")
        return self._full_pipeline(image, show_steps=show_steps)

    def predict_from_array(self, image_array):
        """
        Accept raw numpy array (BGR uint8).
        USE ON: ESP32 (image captured -> sent as array) / RPi camera
        Returns lightweight dict: predicted_class + confidence only.
        """
        result = self._full_pipeline(image_array, show_steps=False)
        return {
            'predicted_class': result['predicted_class'],
            'confidence':      result['confidence']
        }

    # Backward-compatible alias
    def predict_single_image_from_array(self, image_array):
        """Alias for predict_from_array() - keeps old code working."""
        return self.predict_from_array(image_array)

    def run_inference_on_batch(self, images_tensor):
        """
        Run model on a tf.data batch tensor.
        USE ON: Colab notebook Step 8.
        images_tensor: float32 (B, H, W, 3) normalised [0,1]
        Returns: predictions array (B, num_classes)
        """
        return self.model.predict(images_tensor, verbose=0)

    # ── Internal pipeline ─────────────────────────────────────────────
    def _full_pipeline(self, image_bgr, show_steps=False):
        """Runs all 3 steps and returns complete result dict."""
        original = image_bgr.copy()

        # STEP 1: Pre-processing
        preprocessed = self.preprocessor.preprocess_image(image_bgr)

        # STEP 2: Segmentation
        seg = self.segmenter.segment_image(preprocessed)

        # STEP 3: CNN Prediction
        model_input   = preprocessed.astype('float32') / 255.0
        model_input   = np.expand_dims(model_input, axis=0)
        predictions   = self.model.predict(model_input, verbose=0)
        predicted_idx = int(np.argmax(predictions[0]))
        confidence    = float(predictions[0][predicted_idx] * 100)

        predicted_class = (self.class_names[predicted_idx]
                           if predicted_idx < len(self.class_names)
                           else f'Class_{predicted_idx}')

        # Grad-CAM
        heatmap = overlay = None
        if self.grad_cam is not None:
            try:
                heatmap = self.grad_cam.compute_heatmap(model_input, predicted_idx)
                overlay = self.grad_cam.overlay(preprocessed, heatmap)
            except Exception:
                pass

        result = {
            'original':        original,
            'preprocessed':    preprocessed,
            'segmented':       seg['segmented'],
            'masked':          seg['masked'],
            'binary':          seg['binary'],
            'edges':           seg['edges'],
            'predicted_class': predicted_class,
            'predicted_idx':   predicted_idx,
            'confidence':      confidence,
            'all_probs':       predictions[0],
            'heatmap':         heatmap,
            'overlay':         overlay,
        }

        if show_steps:
            self._visualise_steps(result)

        return result

    # =========================================================================
    # VISUALISATION METHODS  (Colab-friendly, display inline)
    # =========================================================================

    def _visualise_steps(self, result):
        """Show all 3 processing steps in one figure."""
        titles = ['1. Original', '2. Pre-processed', '3. K-means Seg.',
                  '4. Diseased Mask', '5. Otsu Binary', '6. Grad-CAM']
        images = [
            result['original'],
            result['preprocessed'],
            result['segmented'],
            result['masked'],
            result['binary'],
            result['overlay'] if result['overlay'] is not None else result['preprocessed']
        ]

        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        axes = axes.flatten()

        for i, (title, img) in enumerate(zip(titles, images)):
            if img is None:
                axes[i].text(0.5, 0.5, 'N/A', ha='center', va='center')
            elif len(img.shape) == 2:
                axes[i].imshow(img, cmap='gray')
            else:
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            if i == 5:
                title = (f"Prediction: {result['predicted_class']}\\n"
                         f"Confidence: {result['confidence']:.1f}%")

            axes[i].set_title(title, fontsize=11, fontweight='bold')
            axes[i].axis('off')

        fig.suptitle('Pest Detection - All 3 Processing Steps',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def show_predictions(self, images_tensor, labels_tensor, class_names=None):
        """
        Show 9 sample predictions from a Colab val_ds batch.
        USE ON: Colab notebook Step 8.

        images_tensor : float32 tensor (B, H, W, 3) normalised [0,1]
        labels_tensor : one-hot tensor (B, num_classes)
        class_names   : list of class name strings (optional override)
        """
        if class_names is not None:
            self.class_names = class_names

        predictions = self.run_inference_on_batch(images_tensor)
        n           = min(9, len(images_tensor))

        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        axes = axes.flatten()

        for i in range(n):
            img        = images_tensor[i].numpy()
            true_idx   = int(np.argmax(labels_tensor[i]))
            pred_idx   = int(np.argmax(predictions[i]))
            confidence = float(predictions[i][pred_idx] * 100)

            true_name = (self.class_names[true_idx]
                         if true_idx < len(self.class_names) else f'Class_{true_idx}')
            pred_name = (self.class_names[pred_idx]
                         if pred_idx < len(self.class_names) else f'Class_{pred_idx}')

            axes[i].imshow(img)
            color = 'green' if true_idx == pred_idx else 'red'
            axes[i].set_title(
                f"True: {true_name}\\nPred: {pred_name}\\n({confidence:.1f}%)",
                color=color, fontsize=9, fontweight='bold'
            )
            axes[i].axis('off')

        for j in range(n, 9):
            axes[j].axis('off')

        correct   = mpatches.Patch(color='green', label='Correct prediction')
        incorrect = mpatches.Patch(color='red',   label='Wrong prediction')
        fig.legend(handles=[correct, incorrect],
                   loc='lower center', ncol=2, fontsize=11)

        fig.suptitle('Sample Predictions - Pest Detection',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0.04, 1, 1])
        plt.show()

    def plot_training_history(self, history):
        """
        Plot accuracy and loss curves after training.
        USE ON: Colab notebook Step 6.
        """
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Accuracy
        axes[0].plot(history.history['accuracy'],     label='Train',      linewidth=2)
        axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0].axhline(y=0.986, color='red', linestyle='--', label='Target 98.6%')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title('Model Accuracy - TinyLiteNet', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Loss
        axes[1].plot(history.history['loss'],     label='Train',      linewidth=2)
        axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Model Loss - TinyLiteNet', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved: training_history.png")

    def show_confusion_matrix(self, val_ds, class_names=None):
        """
        Generate and display confusion matrix from a tf.data dataset.
        USE ON: Colab after training to evaluate model.
        """
        try:
            import seaborn as sns
        except ImportError:
            print("seaborn not installed. Run: pip install seaborn")
            return

        from sklearn.metrics import confusion_matrix, classification_report

        if class_names is not None:
            self.class_names = class_names

        y_true, y_pred = [], []
        for images, labels in val_ds:
            preds  = self.run_inference_on_batch(images)
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            y_pred.extend(np.argmax(preds, axis=1))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.class_names,
                    yticklabels=self.class_names)
        plt.xlabel('Predicted', fontsize=12)
        plt.ylabel('True',      fontsize=12)
        plt.title('Confusion Matrix - Pest & Disease Detection',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Saved: confusion_matrix.png")
        print("\\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=self.class_names))


# ============================================================================
# QUICK SELF-TEST
# ============================================================================

if __name__ == '__main__':
    print("=" * 65)
    print("  IMPROVED PEST DETECTION SYSTEM - SELF TEST")
    print("=" * 65)

    system = PestDetectionSystem(num_classes=9)
    system.compile_model()

    info = system.get_model_info()
    print(f"\\n  Model size   : {info['model_size_mb']} MB   (target <= 1.5 MB)")
    print(f"  Parameters   : {info['parameters_millions']}M   (target <= 2M)")

    dummy_bgr    = np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8)
    preprocessed = ImagePreprocessor.preprocess_image(dummy_bgr)
    print(f"\\n  Pre-processing : {dummy_bgr.shape} -> {preprocessed.shape}  OK")

    seg = ImageSegmenter.segment_image(preprocessed)
    print(f"  Segmentation   : keys = {list(seg.keys())}  OK")

    result = system.predict_from_array(dummy_bgr)
    print(f"\\n  Inference test :")
    print(f"    Predicted class : {result['predicted_class']}")
    print(f"    Confidence      : {result['confidence']:.1f}%")

    print("\\n" + "=" * 65)
    print("  All tests passed - system is ready!")
    print("=" * 65)
''')
    print("✅ Created improved_pest_detection.py")
    
# Now import
from improved_pest_detection import (
    PestDetectionSystem,
    TinyLiteNet,
    ImagePreprocessor,
    ImageSegmenter,
)
print("✅ improved_pest_detection.py imported successfully.")


# ── 2b. Custom callback — saves epoch number after EVERY epoch ───────────────
class EpochSaver(keras.callbacks.Callback):
    """
    Saves last completed epoch to a text file after every epoch.
    Even if next epoch crashes, this file exists → resume works correctly.
    """
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath

    def on_epoch_end(self, epoch, logs=None):
        with open(self.filepath, 'w') as f:
            f.write(str(epoch + 1))  # epoch is 0-based, save as 1-based


# ── 3. Hardware info ──────────────────────────────────────────────────────────
def show_hardware_info():
    print("\n" + "="*65)
    print("  HARDWARE INFO")
    print("="*65)

    gpus = tf.config.list_physical_devices('GPU')
    cpus = tf.config.list_physical_devices('CPU')

    if gpus:
        print(f"  ✅ GPU detected  : {gpus[0].name}")
        print(f"     Training will be FAST (30–60 min)")
        # Allow GPU memory growth to avoid crash
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print(f"  ⚠️  No GPU detected — using CPU only")
        print(f"     Training will be SLOW (1–3 hours)")
        print(f"     Tip: Reduce BATCH_SIZE to 8 if it crashes")

    print(f"  CPUs : {len(cpus)}")
    print(f"  TensorFlow version : {tf.__version__}")
    print("="*65 + "\n")

show_hardware_info()


# ── 4. Dataset download (Kaggle) ──────────────────────────────────────────────
def download_kaggle_dataset():
    """Download dataset from Kaggle automatically."""
    print("="*65)
    print("  DOWNLOADING DATASET FROM KAGGLE")
    print("="*65)

    # Check kaggle.json exists
    kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')

    if not os.path.exists(kaggle_json):
        print("\n❌  kaggle.json not found at:", kaggle_json)
        print("\n   How to get kaggle.json:")
        print("   1. Go to  https://www.kaggle.com")
        print("   2. Click your profile picture → Settings")
        print("   3. Scroll to API section")
        print("   4. Click 'Create New API Token'")
        print("   5. Download kaggle.json")
        print(f"  6. Move it to:  {os.path.expanduser('~/.kaggle/')}")
        print("\n   On Windows create folder:  C:\\Users\\YourName\\.kaggle\\")
        print("   Then place kaggle.json inside it.\n")
        sys.exit(1)

    # Set permissions on Mac/Linux
    if os.name != 'nt':
        os.chmod(kaggle_json, 0o600)

    # Import kaggle
    try:
        import kaggle
    except ImportError:
        print("❌  kaggle package not installed. Run:  pip install kaggle")
        sys.exit(1)

    os.makedirs(DATASET_PATH, exist_ok=True)

    print(f"\n  Dataset  : {KAGGLE_DATASET}")
    print(f"  Save to  : {os.path.abspath(DATASET_PATH)}")
    print(f"\n  Downloading... (this may take 5–10 minutes)\n")

    import subprocess
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
         '--unzip', '-p', DATASET_PATH],
        capture_output=False
    )

    if result.returncode != 0:
        print("\n❌  Download failed. Check your internet and kaggle.json.")
        sys.exit(1)

    print(f"\n✅ Dataset downloaded to: {os.path.abspath(DATASET_PATH)}")


# ── 5. Find the actual class folder inside downloaded dataset ──────────────────
def find_dataset_root(base_path):
    """
    Kaggle datasets often have nested folders.
    Walk down until we find a folder with sub-folders (classes).
    """
    for root, dirs, files in os.walk(base_path):
        # Filter hidden folders
        dirs_clean = [d for d in dirs if not d.startswith('.')]
        if len(dirs_clean) >= 2:
            # Check sub-folders contain images
            for d in dirs_clean[:3]:
                sub = os.path.join(root, d)
                imgs = [f for f in os.listdir(sub)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                if imgs:
                    return root
    return base_path


# ── 6. Load dataset ───────────────────────────────────────────────────────────
def load_dataset(dataset_root):
    """Load images using Keras utility — memory efficient."""
    print("="*65)
    print("  LOADING DATASET")
    print("="*65)
    print(f"  Path     : {os.path.abspath(dataset_root)}")
    print(f"  Classes  : {sorted(os.listdir(dataset_root))}\n")

    # Training split
    train_ds = keras.utils.image_dataset_from_directory(
        dataset_root,
        validation_split=0.2,
        subset='training',
        seed=42,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    # Validation split
    val_ds = keras.utils.image_dataset_from_directory(
        dataset_root,
        validation_split=0.2,
        subset='validation',
        seed=42,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    class_names  = train_ds.class_names
    num_classes  = len(class_names)

    print(f"\n  ✅ Dataset loaded!")
    print(f"     Classes    : {num_classes}")
    print(f"     Class list : {class_names}")

    # ── Preprocessing pipeline ──
    # Step 1: Normalize pixel values to [0, 1]
    normalize = keras.layers.Rescaling(1.0 / 255)

    # Step 2: Data augmentation (training only — matches Paper 1 Table 5)
    augment = keras.Sequential([
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomRotation(0.2),
        keras.layers.RandomZoom(0.3),
        keras.layers.RandomTranslation(0.3, 0.3),
        keras.layers.RandomBrightness(0.2),        # Paper 1: brightness [0.8, 1.2]
    ], name='data_augmentation')

    # CORRECT PIPELINE ORDER:
    # 1. Normalize first (on both train and val)
    train_ds = train_ds.map(
        lambda x, y: (normalize(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    val_ds = val_ds.map(
        lambda x, y: (normalize(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 2. Cache AFTER normalize but BEFORE augmentation
    #    This stores normalized images in memory once.
    #    IMPORTANT: if cache is AFTER augmentation, the same random augmented
    #    images repeat every epoch — model sees less variety, accuracy drops.
    train_ds = train_ds.cache()
    val_ds   = val_ds.cache()

    # 3. Augment AFTER cache (runs fresh random augmentation every epoch)
    train_ds = train_ds.map(
        lambda x, y: (augment(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 4. Prefetch last (prepares next batch while current batch trains)
    train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    val_ds   = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

    return train_ds, val_ds, class_names, num_classes


# ── 7. Build and compile model ────────────────────────────────────────────────
def build_and_compile_model(num_classes):
    """Build Tiny-LiteNet and compile for training."""
    print("="*65)
    print("  BUILDING TINY-LITENET MODEL")
    print("="*65)

    system = PestDetectionSystem(num_classes=num_classes)
    system.compile_model(learning_rate=LEARNING_RATE)

    info = system.get_model_info()
    print(f"\n  Model size   : {info['model_size_mb']} MB   (paper target: 1.2 MB)")
    print(f"  Parameters   : {info['parameters_millions']}M   (paper target: 1.48 M)")
    print(f"  Input shape  : {info['input_shape']}")
    print(f"  Classes      : {num_classes}")

    system.model.summary(line_length=65, print_fn=lambda x: print('  ' + x))

    return system


# ── 8. Build callbacks ────────────────────────────────────────────────────────
def build_callbacks(output_dir):
    """Training callbacks: early stop, reduce LR, checkpoint."""
    os.makedirs(output_dir, exist_ok=True)

    best_model_path = os.path.join(output_dir, 'tinylitenet_best.keras')

    callbacks = [
        # ✅ Saves epoch number after every epoch — critical for resume
        EpochSaver(
            filepath=os.path.join(output_dir, 'last_epoch.txt')
        ),
        # ✅ Saves full model after every epoch
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(output_dir, 'resume_checkpoint.keras'),
            monitor='val_accuracy',
            save_best_only=False,
            verbose=1
        ),
        # Stop if no improvement for 15 epochs
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=15,
            restore_best_weights=True,
            verbose=1
        ),
        # Reduce learning rate when stuck
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=8,
            min_lr=1e-7,
            verbose=1
        ),
        # Save best model automatically
        keras.callbacks.ModelCheckpoint(
            filepath=best_model_path,
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        # Live training log to CSV (so you can check progress)
        keras.callbacks.CSVLogger(
            os.path.join(output_dir, 'training_log.csv'),
            append=False
        ),
    ]

    print(f"\n  ✅ Callbacks ready.")
    print(f"     Best model will be saved to: {best_model_path}")

    return callbacks, best_model_path


# ── 9. Train model ────────────────────────────────────────────────────────────
def train_model(system, train_ds, val_ds, callbacks, initial_epoch=0):
    """Run model.fit and return history."""
    print("\n" + "="*65)
    print("  STARTING TRAINING")
    print("="*65)
    print(f"  Max epochs   : {EPOCHS}  (early stopping may end sooner)")
    print(f"  Batch size   : {BATCH_SIZE}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print("="*65 + "\n")

    start = time.time()

    history = system.model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        initial_epoch=initial_epoch,   # ✅ starts from correct epoch
        callbacks=callbacks,
        verbose=1
    )

    elapsed = time.time() - start
    minutes = int(elapsed // 60)
    seconds = int(elapsed % 60)

    print(f"\n  ⏱  Training time: {minutes} min {seconds} sec")
    return history


# ── 10. Save outputs ──────────────────────────────────────────────────────────
def save_outputs(system, class_names, history, output_dir):
    """Save model, class names, and training plot."""
    os.makedirs(output_dir, exist_ok=True)

    # Save final model
    final_path = os.path.join(output_dir, 'tinylitenet_final.keras')
    system.model.save(final_path)
    print(f"\n  ✅ Final model saved  : {final_path}")

    # Save class names
    json_path = os.path.join(output_dir, 'class_names.json')
    with open(json_path, 'w') as f:
        json.dump(class_names, f, indent=2)
    print(f"  ✅ Class names saved  : {json_path}")

    # Save training plot
    _save_training_plot(history, output_dir)

    print(f"\n  📂 All outputs in: {os.path.abspath(output_dir)}/")


def _save_training_plot(history, output_dir):
    """Plot and save accuracy + loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train',      linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].axhline(y=0.986, color='red', linestyle='--', label='Target 98.6%', alpha=0.7)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy — Tiny-LiteNet', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train',      linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss — Tiny-LiteNet', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(output_dir, 'training_history.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✅ Training plot saved : {plot_path}")


# ── 11. Print final summary ───────────────────────────────────────────────────
def print_final_summary(history, class_names, output_dir):
    """Print final accuracy and what to do next."""
    final_train_acc = history.history['accuracy'][-1]   * 100
    final_val_acc   = history.history['val_accuracy'][-1] * 100
    best_val_acc    = max(history.history['val_accuracy']) * 100

    print("\n" + "="*65)
    print("  TRAINING COMPLETE!")
    print("="*65)
    print(f"  Final training accuracy   : {final_train_acc:.2f}%")
    print(f"  Final validation accuracy : {final_val_acc:.2f}%")
    print(f"  Best validation accuracy  : {best_val_acc:.2f}%")
    print(f"  Paper target              : 98.60%")
    print("="*65)

    print(f"\n  Classes trained ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"    [{i:2d}] {name}")

    print(f"\n  📦 Files ready in  '{output_dir}/':")
    print(f"    tinylitenet_best.keras    <- Use this on Raspberry Pi")
    print(f"    tinylitenet_final.keras   <- Final epoch model")
    print(f"    class_names.json          <- Class labels")
    print(f"    training_log.csv          <- Epoch-by-epoch log")
    print(f"    training_history.png      <- Accuracy/loss plot")

    print(f"\n  NEXT STEPS:")
    print(f"    1. Test: python test_on_image.py --image leaf.jpg")
    print(f"    2. Deploy: copy tinylitenet_best.keras to Raspberry Pi")
    print(f"    3. Use improved_pest_detection.py for inference anywhere")
    print("="*65 + "\n")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    global OUTPUT_DIR  # allow updating the global variable

    print("\n" + "="*65)
    print("  PEST DETECTION — LOCAL LAPTOP TRAINING")
    print("  Tiny-LiteNet  |  Based on Nyakuri et al. 2025")
    print("="*65)

    # In Kaggle notebook, OUTPUT_DIR is already set
    # Just make sure it exists
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"\n  ✅ Output folder: {OUTPUT_DIR}")

    # Step 1: Download dataset if needed
    if USE_KAGGLE:
        if os.path.exists(DATASET_PATH) and len(os.listdir(DATASET_PATH)) > 0:
            print(f"\n  ✅ Dataset folder '{DATASET_PATH}' already exists. Skipping download.")
            print(f"     Delete '{DATASET_PATH}' to re-download.\n")
        else:
            download_kaggle_dataset()
    else:
        if not os.path.exists(DATASET_PATH):
            print(f"\n❌  Dataset folder not found: {os.path.abspath(DATASET_PATH)}")
            print(f"   Create the folder with class sub-folders and images.\n")
            sys.exit(1)
        print(f"  ✅ Using manual dataset at: {os.path.abspath(DATASET_PATH)}\n")

    # Step 2: Find dataset root (handle nested folders from Kaggle)
    dataset_root = find_dataset_root(DATASET_PATH)
    print(f"  📂 Dataset root found: {os.path.abspath(dataset_root)}")

    # Step 3: Load dataset
    train_ds, val_ds, class_names, num_classes = load_dataset(dataset_root)

    # Step 4: Build model
    system = build_and_compile_model(num_classes)
    system.class_names = class_names

    # Add AFTER build_and_compile_model()
    # ✅ FIXED Resume Code
    checkpoint_path = os.path.join(OUTPUT_DIR, 'resume_checkpoint.keras')
    epoch_file      = os.path.join(OUTPUT_DIR, 'last_epoch.txt')
    initial_epoch   = 0  # default = start fresh

    if os.path.exists(checkpoint_path):
        print("\n  ✅ Found checkpoint! Resuming...")
        system.model = keras.models.load_model(checkpoint_path)

        # Read which epoch it stopped at
        if os.path.exists(epoch_file):
            with open(epoch_file, 'r') as f:
                initial_epoch = int(f.read().strip())
            print(f"     Resuming from epoch {initial_epoch + 1}\n")
        else:
            print("     Loaded weights. Starting from epoch 1\n")
    else:
        print("\n  ℹ️  No checkpoint. Starting fresh.\n")

    # Step 5: Setup callbacks
    callbacks, best_model_path = build_callbacks(OUTPUT_DIR)

    # Step 6: Train
    history = train_model(system, train_ds, val_ds, callbacks, initial_epoch)
    # ✅ Save last completed epoch number
    epoch_file = os.path.join(OUTPUT_DIR, 'last_epoch.txt')
    last_epoch = initial_epoch + len(history.history['accuracy'])
    with open(epoch_file, 'w') as f:
        f.write(str(last_epoch))
    print(f"  ✅ Saved epoch count: {last_epoch}")

    # Step 7: Save everything
    save_outputs(system, class_names, history, OUTPUT_DIR)

    # Step 8: Print summary
    print_final_summary(history, class_names, OUTPUT_DIR)


if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# IMPROVED DIAGNOSTIC CELL
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import Counter
import pathlib

print("🔍 RUNNING IMPROVED DIAGNOSTICS...")
print("="*65)

# Dataset path
DATASET_PATH = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8

# 1. First, let's understand the dataset structure
print("\n📁 EXAMINING DATASET STRUCTURE:")
if os.path.exists(DATASET_PATH):
    # Get class names directly from folders
    class_folders = sorted([d for d in os.listdir(DATASET_PATH) 
                           if os.path.isdir(os.path.join(DATASET_PATH, d))])
    print(f"  Found {len(class_folders)} class folders:")
    for i, folder in enumerate(class_folders[:10]):  # Show first 10
        # Count images in this folder
        folder_path = os.path.join(DATASET_PATH, folder)
        img_count = len([f for f in os.listdir(folder_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"    [{i:2d}] {folder[:30]:30} : {img_count} images")
    if len(class_folders) > 10:
        print(f"    ... and {len(class_folders)-10} more classes")
    
    class_names = class_folders
else:
    print(f"❌ Dataset path not found: {DATASET_PATH}")
    class_names = None

# 2. Load dataset properly
print("\n📊 LOADING DATASET:")
try:
    # Load training dataset
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_PATH,
        validation_split=0.2,
        subset='training',
        seed=42,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    
    # Load validation dataset
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_PATH,
        validation_split=0.2,
        subset='validation',
        seed=42,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    
    # Get class names from the dataset
    class_names = train_ds.class_names
    print(f"\n  ✅ Loaded {len(class_names)} classes:")
    for i, name in enumerate(class_names[:10]):
        print(f"     {i:2d}: {name}")
    if len(class_names) > 10:
        print(f"     ... and {len(class_names)-10} more")
    
    # Normalize
    normalize = tf.keras.layers.Rescaling(1.0/255)
    train_ds = train_ds.map(lambda x, y: (normalize(x), y))
    val_ds = val_ds.map(lambda x, y: (normalize(x), y))
    
    # Cache and prefetch
    train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
    
    print("  ✅ Dataset normalized and cached")
    
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    train_ds = None
    val_ds = None

# 3. Check class balance
if train_ds is not None:
    print("\n📊 CLASS DISTRIBUTION (first 500 samples):")
    class_counts = Counter()
    total = 0
    for images, labels in train_ds.unbatch().take(500):
        class_idx = np.argmax(labels.numpy())
        class_counts[class_idx] += 1
        total += 1
    
    for i in range(len(class_names)):
        count = class_counts.get(i, 0)
        pct = (count/total)*100 if total>0 else 0
        bar = '█' * int(pct/2)
        print(f"  {i:2d} {class_names[i][:25]:25}: {count:3} samples ({pct:5.1f}%) {bar}")

# 4. Check for saved model
model_path = '/kaggle/working/trained_models/resume_checkpoint.keras'
if os.path.exists(model_path):
    print(f"\n✅ Found checkpoint: {model_path}")
    model = tf.keras.models.load_model(model_path)
    
    # Test on a few validation images
    if val_ds is not None:
        print("\n📸 MODEL PERFORMANCE ON SAMPLE IMAGES:")
        for images, labels in val_ds.take(1):
            preds = model.predict(images[:5], verbose=0)
            
            fig, axes = plt.subplots(1, 5, figsize=(15, 3))
            for i in range(5):
                true_idx = np.argmax(labels[i])
                pred_idx = np.argmax(preds[i])
                conf = np.max(preds[i]) * 100
                
                axes[i].imshow(images[i])
                color = 'green' if true_idx == pred_idx else 'red'
                axes[i].set_title(f"True: {class_names[true_idx][:10]}\n"
                                 f"Pred: {class_names[pred_idx][:10]}\n"
                                 f"({conf:.1f}%)", color=color, fontsize=9)
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # Show prediction probabilities for first image
            print(f"\n  Prediction probabilities for first image:")
            probs = preds[0]
            top5 = np.argsort(probs)[-5:][::-1]
            for idx in top5:
                print(f"    {class_names[idx][:25]:25}: {probs[idx]*100:5.1f}%")
else:
    print(f"\n❌ No checkpoint found at: {model_path}")
    print("   Training hasn't saved any model yet or is in early epochs.")

# 5. Visual check of training images
if train_ds is not None:
    print("\n🖼️  SAMPLE TRAINING IMAGES:")
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    
    for images, labels in train_ds.take(1):
        for i in range(min(8, len(images))):
            axes[i].imshow(images[i])
            class_idx = np.argmax(labels[i])
            axes[i].set_title(f"{class_names[class_idx][:20]}")
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# 6. Quick check for obvious issues
print("\n🔍 QUICK ISSUE CHECK:")
print("-" * 40)

if train_ds is not None:
    # Check image range
    for images, _ in train_ds.take(1):
        img_min = images.numpy().min()
        img_max = images.numpy().max()
        print(f"  Image value range: [{img_min:.3f}, {img_max:.3f}]")
        if img_min < 0 or img_max > 1:
            print("  ⚠️  Images not properly normalized to [0,1]")
        else:
            print("  ✅ Images properly normalized")

# Check number of classes
if class_names:
    print(f"  Number of classes: {len(class_names)}")
    if len(class_names) != 15:
        print(f"  ⚠️  Expected 15 classes, found {len(class_names)}")
    else:
        print("  ✅ Correct number of classes")

print("\n" + "="*65)
print("✅ DIAGNOSTIC COMPLETE")
print("="*65)

# Recommendations based on findings
print("\n💡 RECOMMENDATIONS:")
print("   Based on your 15% accuracy at epoch 5:")
print("   1. Check if images look correct in the samples above")
print("   2. Make sure all 15 classes have balanced representation")
print("   3. Try these settings in your training cell:")
print("      LEARNING_RATE = 0.0003  (lower than current)")
print("      BATCH_SIZE = 16         (try larger batch)")
print("      In augmentation, use only:")
print("        keras.layers.RandomFlip('horizontal')")
print("        keras.layers.RandomRotation(0.1)")
print("   4. Remove other augmentations temporarily")

In [4]:
# ============================================================
# OPTIMIZED TRAINING CELL - RUN THIS NOW
# ============================================================

import os
import tensorflow as tf
from tensorflow import keras
import numpy as np

# ============================================================
# OPTIMIZED SETTINGS
# ============================================================

DATASET_PATH = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'
OUTPUT_DIR = '/kaggle/working/trained_models'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16  # Increased from 8
LEARNING_RATE = 0.0003  # Lower than default
EPOCHS = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {OUTPUT_DIR}")

# ============================================================
# LOAD DATASET
# ============================================================

print("\n📊 Loading dataset...")
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"✅ Loaded {num_classes} classes: {class_names}")

# ============================================================
# NORMALIZE AND AUGMENT (SIMPLIFIED)
# ============================================================

normalize = keras.layers.Rescaling(1.0/255)

# SIMPLIFIED AUGMENTATION - just flip and slight rotation
augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),  # Reduced from 0.2
], name='augmentation')

# Apply normalization
train_ds = train_ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)

# Cache
train_ds = train_ds.cache()
val_ds = val_ds.cache()

# Apply augmentation only to training
train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print("✅ Data pipeline ready")

# ============================================================
# BUILD MODEL
# ============================================================

print("\n🏗️ Building TinyLiteNet...")

# We need to import the classes from your file
from improved_pest_detection import PestDetectionSystem, TinyLiteNet

system = PestDetectionSystem(num_classes=num_classes)
system.compile_model(learning_rate=LEARNING_RATE)
system.class_names = class_names

# Print model info
info = system.get_model_info()
print(f"  Model size: {info['model_size_mb']} MB")
print(f"  Parameters: {info['parameters_millions']}M")

# ============================================================
# CALLBACKS
# ============================================================

callbacks = [
    # Save best model
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, 'tinylitenet_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Save checkpoint every epoch
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, 'checkpoint_epoch_{epoch}.keras'),
        verbose=0
    ),
    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce LR on plateau
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    # CSV logger
    keras.callbacks.CSVLogger(
        os.path.join(OUTPUT_DIR, 'training_log.csv')
    )
]

print("✅ Callbacks ready")

# ============================================================
# TRAIN
# ============================================================

print("\n🚀 STARTING TRAINING...")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max epochs: {EPOCHS}")
print("="*50)

history = system.model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# SAVE FINAL MODEL
# ============================================================

final_path = os.path.join(OUTPUT_DIR, 'tinylitenet_final.keras')
system.model.save(final_path)
print(f"\n✅ Final model saved to: {final_path}")

# Save class names
import json
with open(os.path.join(OUTPUT_DIR, 'class_names.json'), 'w') as f:
    json.dump(class_names, f, indent=2)

print("\n✅ Training complete! Models saved in:", OUTPUT_DIR)

✅ Output directory: /kaggle/working/trained_models

📊 Loading dataset...
Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
✅ Loaded 15 classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
✅ Data pipeline ready

🏗️ Building TinyLiteNet...
Building new TinyLiteNet model...
  Model size: 1.77 MB
  Parameters: 0.46M
✅ Callbacks ready

🚀 STARTING TRAINING...
  Batch size: 16
  Learning rate: 0.0003
  Max epochs: 50
Epoch 1/50
1032/1032 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.5111 - loss: 1.5399 - precision: 0.7917 - recall: 0.

KeyboardInterrupt: 

In [6]:
# Create zip file of trained models
import zipfile

zip_path = '/kaggle/working/trained_models.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, OUTPUT_DIR)
            zipf.write(file_path, arcname)

print(f"✅ Created zip file: {zip_path}")
print("📥 Download from Kaggle: Data tab → /kaggle/working/trained_models.zip")

✅ Created zip file: /kaggle/working/trained_models.zip
📥 Download from Kaggle: Data tab → /kaggle/working/trained_models.zip


In [5]:
# URGENT: Save your NEW CHAMPION (99.10%)!
import shutil
import zipfile
import os
from datetime import datetime

best_model = '/kaggle/working/trained_models/tinylitenet_best.keras'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save with proud name
champion_name = f'/kaggle/working/CHAMPION_99.10_PERCENT_{timestamp}.keras'
shutil.copy(best_model, champion_name)
print(f"✅ NEW CHAMPION SAVED: 99.10%")

# Create deployment zip
zip_path = '/kaggle/working/plant_disease_champion_99.10.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    zipf.write(best_model, 'plant_disease_model_99.10.keras')
    if os.path.exists('/kaggle/working/trained_models/class_names.json'):
        zipf.write('/kaggle/working/trained_models/class_names.json', 'class_names.json')

print(f"\n📦 Deployment zip created: {zip_path}")
print(f"   Size: {os.path.getsize(zip_path)/(1024*1024):.1f} MB")

print("\n" + "="*60)
print("🏆🏆🏆 YOU DID IT! 99.10% ACCURACY! 🏆🏆🏆")
print("="*60)
print("📥 DOWNLOAD NOW from Data tab → /kaggle/working/")
print("="*60)

✅ NEW CHAMPION SAVED: 99.10%

📦 Deployment zip created: /kaggle/working/plant_disease_champion_99.10.zip
   Size: 5.6 MB

🏆🏆🏆 YOU DID IT! 99.10% ACCURACY! 🏆🏆🏆
📥 DOWNLOAD NOW from Data tab → /kaggle/working/


In [8]:
# Split into smaller 50MB chunks
import os
import zipfile

def split_zip(input_file, chunk_size_mb=50):
    chunk_size = chunk_size_mb * 1024 * 1024
    part_num = 1
    
    with open(input_file, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            with open(f'/kaggle/working/trained_models_part{part_num}.zip', 'wb') as chunk_file:
                chunk_file.write(chunk)
            print(f"✅ Created part {part_num}: {os.path.getsize(f'/kaggle/working/trained_models_part{part_num}.zip')/(1024*1024):.1f}MB")
            part_num += 1

# Split your large zip
split_zip('/kaggle/working/trained_models.zip', 50)
print("\n📥 Download each part separately from Data tab!")

✅ Created part 1: 50.0MB
✅ Created part 2: 50.0MB
✅ Created part 3: 50.0MB
✅ Created part 4: 50.0MB
✅ Created part 5: 50.0MB
✅ Created part 6: 0.5MB

📥 Download each part separately from Data tab!
